# Step 2 Diagnostics: GOSH
How GOSH's IDs link (`project_id`), which PICANet file is right (`_eprweights`), how the PICANet and archived files differ, and where the 2014 hole in the lab files sits. Counts only. The decisions these support are used by the manifest (step 3); nothing here changes any data.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import diagnostics as D
import gosh_links as GL
import manifest as M

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
gosh_dir = base / "GOSH"
open_picu_dir = base / "open_picu"
reports_dir = open_picu_dir / "reports"

main = "PICANet_All_patients_SR_011024_projectID.csv"
epr = "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv"
archived = "Archived/picanet_file_with_mapping.csv"
log = D.Log()

## 1. How the IDs link
GOSH links by `project_id`. The supplier's `admission_identifier` is on about 56% of heart-rate rows only.

In [ ]:
with log("1a. How the IDs link"):
    gosh_links = GL.check_links(
        gosh_dir,
        picanet=gosh_dir / main, picanet_epr=gosh_dir / epr, archived=gosh_dir / archived,
        hr_file=gosh_dir / "merged_values" / "HR_merged.csv",
        parquet=gosh_dir / "gosh_mech_vent_31March2026/final_gosh_dataset.parquet",
    )

In [ ]:
with log("1b. Windows"):
    GL.check_windows(gosh_dir, archived=archived,
                     parquet="gosh_mech_vent_31March2026/final_gosh_dataset.parquet", links=gosh_links)

## 2. Which PICANet file
The main file and `_eprweights` hold the same rows; the archived file differs in times, PIM and discharge status.

In [ ]:
with log("2a. Which PICANet file"):
    cmp = GL.compare_picanet(gosh_dir, picanet=main, picanet_epr=epr, archived=archived)

In [ ]:
with log("2b. Outcomes, start, end, PIM"):
    oc = GL.compare_outcomes(gosh_dir, picanet=main, archived=archived)

In [ ]:
with log("2c. Discharge status"):
    st = GL.compare_status(gosh_dir, picanet=main, archived=archived)

## 3. Vitals coverage
How many PICANet admissions have vitals, by supplier identifier and by time window, against the archived windows and against the main-file windows (which the manifest uses).

In [ ]:
vit_files = [f"merged_values/{n}_merged.csv" for n in (
    "HR", "PulseRate", "RespiratoryRate", "SpO2", "Temperature",
    "Invasive_systolic_BP", "Invasive_diastolic_BP", "Invasive_mean_BP",
    "Non-Invasive_systolic_BP", "Non-Invasive_diastolic_BP", "Non-Invasive_mean_BP")]
log.add("3a. Vitals coverage against the archived windows",
        GL.vitals_coverage(gosh_dir, archived=archived, files=vit_files))

In [ ]:
log.add("3b. Vitals coverage against the main-file windows",
        GL.vitals_coverage(gosh_dir, archived=epr, start_col="AdDateTime", end_col="UnitDisDateTime", files=vit_files))

## 4. The 2014 hole in the lab files
The flowsheet lab files have no rows in 2014, and haemoglobin has about half the usual rows in 2015. Not an Epic effect (that is 2019).

In [ ]:
log.add("4a. Haemoglobin rows by month",
        D.rows_by_month(gosh_dir / "merged_lab_values/merged_Hb.csv", "observation_datetime", start="2013-06", end="2015-12"))

In [ ]:
log.add("4b. ALT flowsheet rows by month",
        D.rows_by_month(gosh_dir / "raw_lab_values/lab_variables_ALT_flowsheet_rows.csv", "chart_time", start="2013-06", end="2015-12"))

## 4c. Lab coverage by admission year
For each eligible GOSH stay (the full manifest, before any Epic-era drop), does the stay have at least one row in each lab file? Rows are attached to stays by patient and time, as in the manifest. A share that falls to zero for admissions between October 2013 and June 2015 confirms the hole is in the data and not only in these two files. The same table shows whether anything changes at the Epic switch (2019). Counts and shares only.

In [ ]:
gosh_manifest, gosh_steps = M.gosh_manifest(gosh_dir)
lab_cov = GL.lab_coverage_by_year(gosh_dir, gosh_manifest)
log.add("4c. Share of eligible stays with a row, by lab file and admission year", lab_cov.T)

## 5. Save

In [ ]:
log.write(reports_dir / f"GOSH_diagnostics_{pd.Timestamp.today().date()}.md", title="GOSH diagnostics")